# 📊 Veri Keşfi - Trading Bot Analizi

Bu notebook, trading bot için veri keşfi ve analiz içerir.

**Kapsam:**
- Piyasa verisi analizi
- Coin seçim metrikleri
- Teknik indikatör hesaplamaları
- Korelasyon analizi
- Volatilite ve likidite profilleri

**Faz:** 4 - Coin Selection Agent

**Tarih:** {datetime}


## 🔧 Kurulum ve Import'lar

In [ ]:
# Standard library imports
import sys
import os
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Trading
from binance.client import Client

# Project paths
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Import'lar tamamlandı")
print(f"📁 Proje dizini: {PROJECT_ROOT}")

## 🔑 API Bağlantısı

In [ ]:
# Binance client oluştur (public data için API key gerekmez)
client = Client("", "")

# Test bağlantı
try:
    server_time = client.get_server_time()
    print(f"✅ Binance API bağlantısı başarılı")
    print(f"⏰ Server zamanı: {datetime.fromtimestamp(server_time['serverTime']/1000)}")
except Exception as e:
    print(f"❌ Bağlantı hatası: {e}")

## 📈 1. Piyasa Genel Bakış

In [ ]:
# Tüm USDT çiftlerini al
tickers = client.get_ticker()
usdt_pairs = [t for t in tickers if t['symbol'].endswith('USDT')]

print(f"📊 Toplam USDT çifti: {len(usdt_pairs)}")

# DataFrame'e çevir
df_market = pd.DataFrame(usdt_pairs)
df_market['quoteVolume'] = df_market['quoteVolume'].astype(float)
df_market['priceChangePercent'] = df_market['priceChangePercent'].astype(float)
df_market['lastPrice'] = df_market['lastPrice'].astype(float)

# Sıralama
df_market_sorted = df_market.sort_values('quoteVolume', ascending=False)

# İlk 20'yi göster
print("\n🔝 En Yüksek Hacimli 20 Coin:")
print(df_market_sorted[['symbol', 'lastPrice', 'priceChangePercent', 'quoteVolume']].head(20))

In [ ]:
# Hacim dağılımı görselleştirme
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 30 coin hacim grafiği
top_30 = df_market_sorted.head(30)
axes[0].barh(top_30['symbol'], top_30['quoteVolume'] / 1e6)
axes[0].set_xlabel('24s Hacim (Milyon $)', fontsize=12)
axes[0].set_title('En Yüksek Hacimli 30 Coin', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()

# Fiyat değişim dağılımı
axes[1].hist(df_market['priceChangePercent'], bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Fiyat Değişimi (%)', fontsize=12)
axes[1].set_ylabel('Coin Sayısı', fontsize=12)
axes[1].set_title('24 Saatlik Fiyat Değişim Dağılımı', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n📊 İstatistikler:")
print(f"   Yükseliş: {len(df_market[df_market['priceChangePercent'] > 0])} coin")
print(f"   Düşüş: {len(df_market[df_market['priceChangePercent'] < 0])} coin")
print(f"   Ortalama değişim: {df_market['priceChangePercent'].mean():.2f}%")

## 🎯 2. Coin Seçim Filtreleme

In [ ]:
# Filtreleme kriterleri (mimari belgeye göre)
MIN_VOLUME_USD = 10_000_000  # $10M

# Kara liste
BLACKLIST = ['LUNA', 'LUNC', 'UST', 'USTC', 'BUSD', 'TUSD', 'USDC', 'DAI', 'USDP']

# Filtreleme
df_filtered = df_market[
    (df_market['quoteVolume'] >= MIN_VOLUME_USD) &
    (~df_market['symbol'].str.contains('|'.join(BLACKLIST), case=False))
].copy()

print(f"✅ Filtreleme sonrası coin sayısı: {len(df_filtered)}")
print(f"   Hacim filtresi: >= ${MIN_VOLUME_USD:,.0f}")
print(f"   Kara liste: {len(BLACKLIST)} coin")

# En iyi 50'yi göster
df_filtered_sorted = df_filtered.sort_values('quoteVolume', ascending=False)
print("\n🎯 Filtrelenmiş En İyi 50 Coin:")
print(df_filtered_sorted[['symbol', 'lastPrice', 'priceChangePercent', 'quoteVolume']].head(50))

## 📊 3. Teknik Metrik Hesaplama

In [ ]:
# Örnek coin seç (analiz için)
SAMPLE_SYMBOLS = df_filtered_sorted.head(10)['symbol'].tolist()

print(f"📈 Analiz edilecek coinler: {', '.join(SAMPLE_SYMBOLS)}")

# Her coin için metrik hesapla
metrics_data = []

for symbol in SAMPLE_SYMBOLS:
    try:
        # Kline verisi al (60 gün, günlük)
        klines = client.get_klines(
            symbol=symbol,
            interval=Client.KLINE_INTERVAL_1DAY,
            limit=60
        )
        
        # DataFrame'e çevir
        df_kline = pd.DataFrame(klines, columns=[
            'timestamp', 'open', 'high', 'low', 'close', 'volume',
            'close_time', 'quote_volume', 'trades', 'taker_buy_base',
            'taker_buy_quote', 'ignore'
        ])
        
        # Veri tiplerini düzelt
        for col in ['open', 'high', 'low', 'close', 'volume']:
            df_kline[col] = df_kline[col].astype(float)
        
        # Metrikleri hesapla
        closes = df_kline['close'].values
        highs = df_kline['high'].values
        lows = df_kline['low'].values
        volumes = df_kline['volume'].values
        
        # ATR (basitleştirilmiş)
        tr = np.maximum(
            highs[1:] - lows[1:],
            np.maximum(
                np.abs(highs[1:] - closes[:-1]),
                np.abs(lows[1:] - closes[:-1])
            )
        )
        atr = np.mean(tr[-14:])
        atr_pct = (atr / closes[-1]) * 100
        
        # Volatilite (std)
        returns = np.diff(closes) / closes[:-1]
        volatility = np.std(returns[-30:]) * np.sqrt(365) * 100  # Yıllık %
        
        # RVOL
        recent_vol = np.mean(volumes[-5:])
        avg_vol = np.mean(volumes[-14:])
        rvol = recent_vol / avg_vol if avg_vol > 0 else 1.0
        
        # RSI (basitleştirilmiş)
        deltas = np.diff(closes[-15:])
        gains = np.where(deltas > 0, deltas, 0)
        losses = np.where(deltas < 0, -deltas, 0)
        avg_gain = np.mean(gains)
        avg_loss = np.mean(losses)
        rsi = 100 - (100 / (1 + avg_gain / avg_loss)) if avg_loss > 0 else 100
        
        metrics_data.append({
            'symbol': symbol,
            'atr_pct': round(atr_pct, 2),
            'volatility_annual_pct': round(volatility, 2),
            'rvol': round(rvol, 2),
            'rsi': round(rsi, 2),
            'price': closes[-1],
        })
        
    except Exception as e:
        print(f"⚠️  {symbol} için hata: {e}")
        continue

# DataFrame oluştur
df_metrics = pd.DataFrame(metrics_data)
print("\n📊 Hesaplanan Metrikler:")
print(df_metrics)

In [ ]:
# Metrikleri görselleştir
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ATR
axes[0, 0].bar(df_metrics['symbol'], df_metrics['atr_pct'], color='steelblue')
axes[0, 0].set_title('ATR (%)', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('ATR %', fontsize=12)
axes[0, 0].tick_params(axis='x', rotation=45)

# Volatilite
axes[0, 1].bar(df_metrics['symbol'], df_metrics['volatility_annual_pct'], color='coral')
axes[0, 1].set_title('Yıllık Volatilite (%)', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Volatilite %', fontsize=12)
axes[0, 1].tick_params(axis='x', rotation=45)

# RVOL
axes[1, 0].bar(df_metrics['symbol'], df_metrics['rvol'], color='mediumseagreen')
axes[1, 0].axhline(1.0, color='red', linestyle='--', linewidth=2, label='Baseline')
axes[1, 0].set_title('Relative Volume (RVOL)', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('RVOL', fontsize=12)
axes[1, 0].legend()
axes[1, 0].tick_params(axis='x', rotation=45)

# RSI
axes[1, 1].bar(df_metrics['symbol'], df_metrics['rsi'], color='orchid')
axes[1, 1].axhline(30, color='green', linestyle='--', linewidth=1, label='Oversold')
axes[1, 1].axhline(70, color='red', linestyle='--', linewidth=1, label='Overbought')
axes[1, 1].set_title('RSI (14)', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('RSI', fontsize=12)
axes[1, 1].set_ylim(0, 100)
axes[1, 1].legend()
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 🔗 4. Korelasyon Analizi

In [ ]:
# Her coin için fiyat serisini al
price_data = {}

for symbol in SAMPLE_SYMBOLS:
    try:
        klines = client.get_klines(
            symbol=symbol,
            interval=Client.KLINE_INTERVAL_1DAY,
            limit=30
        )
        
        closes = [float(k[4]) for k in klines]
        price_data[symbol] = closes
        
    except Exception as e:
        print(f"⚠️  {symbol} için hata: {e}")
        continue

# DataFrame oluştur
df_prices = pd.DataFrame(price_data)

# Korelasyon hesapla
correlation_matrix = df_prices.pct_change().corr()

# Görselleştir
plt.figure(figsize=(12, 10))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    square=True,
    linewidths=1,
    cbar_kws={"shrink": 0.8}
)
plt.title('Coin Korelasyon Matrisi (30 Günlük Getiriler)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Korelasyon İstatistikleri:")
print(f"   Ortalama korelasyon: {correlation_matrix.values[np.triu_indices_from(correlation_matrix.values, k=1)].mean():.3f}")
print(f"   Max korelasyon: {correlation_matrix.values[np.triu_indices_from(correlation_matrix.values, k=1)].max():.3f}")
print(f"   Min korelasyon: {correlation_matrix.values[np.triu_indices_from(correlation_matrix.values, k=1)].min():.3f}")

## 📈 5. Detaylı Coin Analizi

In [ ]:
# Tek bir coin için detaylı analiz
ANALYSIS_SYMBOL = SAMPLE_SYMBOLS[0]  # İlk coin

print(f"🔍 Detaylı Analiz: {ANALYSIS_SYMBOL}")

# 90 günlük veri al
klines = client.get_klines(
    symbol=ANALYSIS_SYMBOL,
    interval=Client.KLINE_INTERVAL_1DAY,
    limit=90
)

# DataFrame oluştur
df_coin = pd.DataFrame(klines, columns=[
    'timestamp', 'open', 'high', 'low', 'close', 'volume',
    'close_time', 'quote_volume', 'trades', 'taker_buy_base',
    'taker_buy_quote', 'ignore'
])

for col in ['open', 'high', 'low', 'close', 'volume', 'quote_volume']:
    df_coin[col] = df_coin[col].astype(float)

df_coin['timestamp'] = pd.to_datetime(df_coin['timestamp'], unit='ms')
df_coin.set_index('timestamp', inplace=True)

# Görselleştir
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Fiyat
axes[0].plot(df_coin.index, df_coin['close'], linewidth=2, color='steelblue')
axes[0].fill_between(df_coin.index, df_coin['low'], df_coin['high'], alpha=0.2, color='steelblue')
axes[0].set_title(f'{ANALYSIS_SYMBOL} - Fiyat Hareketi', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Fiyat ($)', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Hacim
axes[1].bar(df_coin.index, df_coin['quote_volume'] / 1e6, color='coral', alpha=0.7)
axes[1].set_title(f'{ANALYSIS_SYMBOL} - İşlem Hacmi', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Hacim (Milyon $)', fontsize=12)
axes[1].grid(True, alpha=0.3)

# Günlük getiri
returns = df_coin['close'].pct_change() * 100
colors = ['green' if r > 0 else 'red' for r in returns]
axes[2].bar(df_coin.index, returns, color=colors, alpha=0.7)
axes[2].axhline(0, color='black', linestyle='--', linewidth=1)
axes[2].set_title(f'{ANALYSIS_SYMBOL} - Günlük Getiri (%)', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Getiri %', fontsize=12)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# İstatistikler
print(f"\n📊 {ANALYSIS_SYMBOL} İstatistikleri (90 gün):")
print(f"   Ortalama fiyat: ${df_coin['close'].mean():.4f}")
print(f"   Min fiyat: ${df_coin['close'].min():.4f}")
print(f"   Max fiyat: ${df_coin['close'].max():.4f}")
print(f"   Ortalama hacim: ${df_coin['quote_volume'].mean():,.0f}")
print(f"   Ortalama günlük getiri: {returns.mean():.2f}%")
print(f"   Getiri std: {returns.std():.2f}%")

## 💾 6. Sonuçları Kaydet

In [ ]:
# Data dizini oluştur
data_dir = PROJECT_ROOT / 'data' / 'processed'
data_dir.mkdir(parents=True, exist_ok=True)

# Filtrelenmiş coin listesini kaydet
output_file = data_dir / f'coin_universe_{datetime.now().strftime("%Y%m%d")}.csv'
df_filtered_sorted.to_csv(output_file, index=False)

print(f"✅ Coin listesi kaydedildi: {output_file}")
print(f"   Toplam coin: {len(df_filtered_sorted)}")

# Metrikleri kaydet
metrics_file = data_dir / f'coin_metrics_{datetime.now().strftime("%Y%m%d")}.csv'
df_metrics.to_csv(metrics_file, index=False)

print(f"✅ Metrikler kaydedildi: {metrics_file}")

## 📝 7. Özet ve Sonuçlar

In [ ]:
print("="*80)
print("📊 VERİ KEŞFİ ÖZET RAPORU")
print("="*80)

print(f"\n🎯 Genel İstatistikler:")
print(f"   Toplam USDT çifti: {len(usdt_pairs)}")
print(f"   Filtrelenmiş coinler: {len(df_filtered_sorted)}")
print(f"   Analiz edilen coinler: {len(SAMPLE_SYMBOLS)}")

print(f"\n📈 Piyasa Durumu:")
print(f"   Yükseliş oranı: {(len(df_market[df_market['priceChangePercent'] > 0]) / len(df_market) * 100):.1f}%")
print(f"   Ortalama fiyat değişimi: {df_market['priceChangePercent'].mean():.2f}%")

print(f"\n🎲 Metrik Aralıkları:")
print(f"   ATR: {df_metrics['atr_pct'].min():.2f}% - {df_metrics['atr_pct'].max():.2f}%")
print(f"   Volatilite: {df_metrics['volatility_annual_pct'].min():.2f}% - {df_metrics['volatility_annual_pct'].max():.2f}%")
print(f"   RVOL: {df_metrics['rvol'].min():.2f} - {df_metrics['rvol'].max():.2f}")
print(f"   RSI: {df_metrics['rsi'].min():.2f} - {df_metrics['rsi'].max():.2f}")

print(f"\n✅ Veri keşfi tamamlandı!")
print(f"⏰ Tarih: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

## 🔜 Sonraki Adımlar

1. **ML Model Geliştirme:** Coin seçimi için LightGBM Ranker modeli eğitimi
2. **Skorlama Sistemi:** 6 kriterli skorlama sisteminin implementasyonu
3. **Portföy Oluşturma:** Dinamik ağırlıklandırma ve yeniden dengeleme
4. **Backtesting:** Coin seçim stratejisinin geçmiş performans testi
5. **Otomasyonu:** Coin seçim ajanının sisteme entegrasyonu

---

**Notebook Sonu**